In [11]:
%pip install requests beautifulsoup4 pandas spacy

Note: you may need to restart the kernel to use updated packages.


In [14]:
import requests
from bs4 import BeautifulSoup
import spacy
import re
import json
import pandas as pd
import unicodedata

url = "https://en.wikipedia.org/wiki/Steve_Jobs"
html = requests.get(url, headers={"User-Agent": "StudentBot/1.0"}).text
soup = BeautifulSoup(html, "html.parser")

article = soup.find("div", id="mw-content-text")
for tag in article.select("table, sup, .mw-editsection, .navbox"):
    tag.decompose()

text = " ".join(p.get_text(" ", strip=True)
                for p in article.find_all("p"))

# Preprocess and normalize
text = unicodedata.normalize("NFKC", text)
text = re.sub(r"\[\d+\]", "", text)
text = re.sub(r"\s+", " ", text).strip()


# Extract named entities
nlp = spacy.load("en_core_web_sm")
doc = nlp(text)

entities = []
for ent in doc.ents:
    if ent.label_ in ["PERSON", "ORG", "GPE", "DATE", "WORK_OF_ART"]:
        entities.append({
            "entity": ent.text,
            "type": ent.label_
        })

# Extract years and decades using regex
for date in set(re.findall(r"\b(?:19|20)\d{2}s?\b", text)):
    entities.append({"entity": date, "type": "DATE"})

# Remove duplicates and create a table
entities = [dict(t) for t in {
    (e["entity"], e["type"]): e for e in entities
}.values()]

df = pd.DataFrame(entities)
print(df.to_string(index=False))

# Save entities to JSON
with open("steve_jobs_entities.json", "w", encoding="utf-8") as f:
    json.dump(entities, f, ensure_ascii=False, indent=4)

print("\nEntities saved to steve_jobs_entities.json")

                                            entity        type
                                  Steven Paul Jobs      PERSON
                                 February 24, 1955        DATE
                                   October 5, 2011        DATE
                                         the 1970s        DATE
                                             1980s        DATE
                                              Jobs      PERSON
                                        Apple Inc.         ORG
                                     Steve Wozniak      PERSON
                            Apple Computer Company         ORG
                                              1976        DATE
                                              1985        DATE
                                              NeXT         ORG
                                     the same year        DATE
                                             Pixar         ORG
                                              1986     